# Load Data

In [1]:
import numpy as np

loaded = np.load("ecg_dataset.npz")
X = loaded["x"]
y = loaded["y"]

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Model Creation

In [10]:
from tensorflow.keras import layers, Model, Input, models

def residual_block(x, filters, kernel_size=3, dropout_rate=0.3):
    shortcut = x

    x = layers.Conv1D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv1D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)


    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    x = layers.Dropout(dropout_rate)(x)

    return x

In [11]:
def build_residual_cnn_feature_extractor_128(input_shape):
    inputs = Input(shape=input_shape)

    x = layers.Conv1D(64, kernel_size=3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x, filters=64, kernel_size=3, dropout_rate=0.3)

    x = residual_block(x, filters=64, kernel_size=3, dropout_rate=0.3)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)
    features = layers.Dense(128, activation='relu')(x)

    feature_model = Model(inputs=inputs, outputs=features, name="ResidualCNN_FeatureExtractor_128")
    return feature_model


In [12]:
def build_residual_cnn_feature_extractor_32(input_shape):
    inputs = Input(shape=input_shape)

    x = layers.Conv1D(32, kernel_size=5, padding='same')(inputs)  # smaller filters
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x, filters=32, kernel_size=5, dropout_rate=0.2)
    x = residual_block(x, filters=32, kernel_size=5, dropout_rate=0.2)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    features = layers.Dense(32, activation='relu')(x)  # Output: 32 features

    model = Model(inputs, features, name="ResidualCNN_FeatureExtractor_32")
    return model

In [13]:
def build_residual_cnn_feature_extractor_64(input_shape):
    inputs = Input(shape=input_shape)

    x = layers.Conv1D(48, kernel_size=3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x, filters=48, kernel_size=3, dropout_rate=0.25)
    x = residual_block(x, filters=48, kernel_size=3, dropout_rate=0.25)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.4)(x)
    features = layers.Dense(64, activation='relu')(x)  # Output: 64 features

    model = Model(inputs, features, name="ResidualCNN_FeatureExtractor_64")
    return model

In [14]:
def build_classifier_from_features(n_features):
    inputs = Input(shape=(n_features,))
    outputs = layers.Dense(1, activation='sigmoid')(inputs)
    model = Model(inputs, outputs, name="ClassificationHead")
    return model

In [15]:
from tensorflow.keras.metrics import Precision, Recall, AUC

feature_model_128 = build_residual_cnn_feature_extractor_128((5000, 12))
classifier_128 = build_classifier_from_features(128)
full_model_128 = models.Sequential([feature_model_128, classifier_128])

full_model_128.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', Precision(name='precision'), Recall(name='recall'), AUC(name='auc')])

In [16]:
from tensorflow.keras.metrics import Precision, Recall, AUC

feature_model_32 = build_residual_cnn_feature_extractor_32((5000, 12))
classifier_32 = build_classifier_from_features(32)
full_model_32 = models.Sequential([feature_model_32, classifier_32])

full_model_32.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', Precision(name='precision'), Recall(name='recall'), AUC(name='auc')])

In [17]:
from tensorflow.keras.metrics import Precision, Recall, AUC

feature_model_64 = build_residual_cnn_feature_extractor_64((5000, 12))
classifier_64 = build_classifier_from_features(64)
full_model_64 = models.Sequential([feature_model_64, classifier_64])

full_model_64.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', Precision(name='precision'), Recall(name='recall'), AUC(name='auc')])

# Train Model

In [18]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
        monitor='val_loss',       # Track validation loss
        patience=3,               # Stop after 3 epochs with no improvement
        restore_best_weights=True
    )

def get_checkpoint(model_name):
    checkpoint = ModelCheckpoint(
        model_name,                      # filename to save
        monitor="val_accuracy",          # what to track
        mode="max",                      # because higher val_accuracy is better
        save_best_only=True,             # only save if it's the best so far
        verbose=1                        # prints info when it saves
    )
    return checkpoint
    

In [19]:
full_model_128.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop, get_checkpoint("ResidualCNN_FeatureExtractor_128.h5")])

Epoch 1/100
378/378 [==============================] - ETA: 0s - loss: 0.4680 - accuracy: 0.7862 - precision: 0.7761 - recall: 0.6518 - auc: 0.8490
Epoch 1: val_accuracy improved from -inf to 0.83951, saving model to ResidualCNN_FeatureExtractor_128.h5
378/378 [==============================] - 591s 2s/step - loss: 0.4680 - accuracy: 0.7862 - precision: 0.7761 - recall: 0.6518 - auc: 0.8490 - val_loss: 0.3752 - val_accuracy: 0.8395 - val_precision: 0.8502 - val_recall: 0.7253 - val_auc: 0.9082
Epoch 2/100


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


378/378 [==============================] - ETA: 0s - loss: 0.4055 - accuracy: 0.8215 - precision: 0.8195 - recall: 0.7084 - auc: 0.8863
Epoch 2: val_accuracy did not improve from 0.83951
378/378 [==============================] - 619s 2s/step - loss: 0.4055 - accuracy: 0.8215 - precision: 0.8195 - recall: 0.7084 - auc: 0.8863 - val_loss: 0.4057 - val_accuracy: 0.8226 - val_precision: 0.7347 - val_recall: 0.8689 - val_auc: 0.9151
Epoch 3/100
378/378 [==============================] - ETA: 0s - loss: 0.3868 - accuracy: 0.8338 - precision: 0.8333 - recall: 0.7290 - auc: 0.8956
Epoch 3: val_accuracy did not improve from 0.83951
378/378 [==============================] - 608s 2s/step - loss: 0.3868 - accuracy: 0.8338 - precision: 0.8333 - recall: 0.7290 - auc: 0.8956 - val_loss: 0.3967 - val_accuracy: 0.8157 - val_precision: 0.9500 - val_recall: 0.5676 - val_auc: 0.9173
Epoch 4/100
378/378 [==============================] - ETA: 0s - loss: 0.3736 - accuracy: 0.8377 - precision: 0.8436 - rec

In [20]:
full_model_32.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop, get_checkpoint("ResidualCNN_FeatureExtractor_32.h5")])

Epoch 1/100
378/378 [==============================] - ETA: 0s - loss: 0.4600 - accuracy: 0.7909 - precision: 0.7865 - recall: 0.6528 - auc: 0.8522
Epoch 1: val_accuracy improved from -inf to 0.83488, saving model to ResidualCNN_FeatureExtractor_32.h5
378/378 [==============================] - 294s 765ms/step - loss: 0.4600 - accuracy: 0.7909 - precision: 0.7865 - recall: 0.6528 - auc: 0.8522 - val_loss: 0.3822 - val_accuracy: 0.8349 - val_precision: 0.7670 - val_recall: 0.8415 - val_auc: 0.9160
Epoch 2/100
378/378 [==============================] - ETA: 0s - loss: 0.3989 - accuracy: 0.8286 - precision: 0.8330 - recall: 0.7132 - auc: 0.8892
Epoch 2: val_accuracy improved from 0.83488 to 0.84745, saving model to ResidualCNN_FeatureExtractor_32.h5
378/378 [==============================] - 280s 740ms/step - loss: 0.3989 - accuracy: 0.8286 - precision: 0.8330 - recall: 0.7132 - auc: 0.8892 - val_loss: 0.3403 - val_accuracy: 0.8475 - val_precision: 0.8819 - val_recall: 0.7129 - val_auc: 0.

In [21]:
full_model_64.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop, get_checkpoint("ResidualCNN_FeatureExtractor_64.h5")])

Epoch 1/100
378/378 [==============================] - ETA: 0s - loss: 0.4661 - accuracy: 0.7929 - precision: 0.7792 - recall: 0.6707 - auc: 0.8504
Epoch 1: val_accuracy improved from -inf to 0.48743, saving model to ResidualCNN_FeatureExtractor_64.h5
378/378 [==============================] - 438s 1s/step - loss: 0.4661 - accuracy: 0.7929 - precision: 0.7792 - recall: 0.6707 - auc: 0.8504 - val_loss: 0.9133 - val_accuracy: 0.4874 - val_precision: 0.4373 - val_recall: 0.9959 - val_auc: 0.8979
Epoch 2/100
378/378 [==============================] - ETA: 0s - loss: 0.3976 - accuracy: 0.8270 - precision: 0.8240 - recall: 0.7199 - auc: 0.8913
Epoch 2: val_accuracy improved from 0.48743 to 0.78921, saving model to ResidualCNN_FeatureExtractor_64.h5
378/378 [==============================] - 421s 1s/step - loss: 0.3976 - accuracy: 0.8270 - precision: 0.8240 - recall: 0.7199 - auc: 0.8913 - val_loss: 0.5411 - val_accuracy: 0.7892 - val_precision: 0.9625 - val_recall: 0.4905 - val_auc: 0.9011
E